In [85]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import re
from collections import Counter
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

english_stopwords = set(stopwords.words('english'))
german_stopwords = set(stopwords.words('german'))

all_stopwords = english_stopwords.union(german_stopwords)

# Stopwords erweitern
all_stopwords.update({
    'game', 'play', 'very', 'good', 'just', 'like', 'really', 'at',
    'this', 'that', 'with', 'have', 'you', 'and', 'the', 'i', 'of', 'a', 'in', 'for', 'be', 'what', 'but', 'it', 'to', 'id',
    'der', 'die', 'das', 'und', 'oder', 'aber', 'ein', 'eine', 'einer',
    'eines', 'dem', 'den', 'des', 'zu', 'zum', 'zur', 'mit', 'auf',
    'für', 'von', 'ist', 'im', 'in', 'am', 'an', 'auch', 'als', 'bei',
    'nicht', 'nur', 'noch', 'so', 'wie', 'man', 'ich', 'du', 'er',
    'sie', 'es', 'wir', 'ihr', 'mich', 'dich', 'mein', 'dein', 'sein',
    'ihr', 'unser', 'euer', 'hat', 'haben', 'war', 'waren', 'wird',
    'werden', 'kann', 'könnte', 'können', 'schon', 'sehr', 'mehr',
    'less', 'game', 'games', 'play', 'playing', 'played', 'player',
    'players', 'just', 'really', 'good', 'great', 'fun', 'like',
    'love', 'get', 'got', 'one', 'would', 'can', 'could', 'much', 'are','on','or','has',
    'many', 'also', 'well', 'even', 'still', 'way', 'make', 'made', 'if', 'my', 'will',
    'time', 'hours', 'hour', 'is', 'game', 'review', 'play', 'see', 'cant', 'Although', 'link', 'linking', 'used', 'may', 'Program', 'times', 'case', 'far', 'something', 'lot'
})

positive_stopwords = {
    "good", "great", "fun", "amazing", "awesome", "love",
    "liked", "favorite", "best", "recommend", "recommended",
    "worth", "enjoy", "enjoyed"
}
generic_review_stopwords = {
    "game", "games", "steam", "review", "reviews", "play",
    "played", "playing", "hours", "hour", "buy", "bought",
    "price", "content", "story", "graphics", "music",
    "multiplayer", "singleplayer"
}
filler_stopwords = {
    "just", "really", "very", "much", "many", "one", "also",
    "even", "still", "would", "could", "can", "get", "got"
}
all_stopwords = english_stopwords.union(german_stopwords)
all_stopwords.update(positive_stopwords)
all_stopwords.update(generic_review_stopwords)
all_stopwords.update(filler_stopwords)

[nltk_data] Downloading package stopwords to /home/sage/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
import os
import datetime as dt
import time
import copy
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd 
import datetime
import requests
import time
import pickle
import json
from sklearn.feature_extraction.text import CountVectorizer


games = set()
allready_done = set()

files = os.listdir("./outputs/")
for filename in files:
    if filename.startswith("review_"):
        games.add(filename.split('eview_')[1].split('.pk')[0])
start = 0
end = len(games)

#games = {25000,211820,239140,244850,305620}
#allready_done = set()
# games = {330460}
#games = {305620}


for game in games:
    to_load = None
    with open(f"./outputs/review_{game}.pk", "rb") as f:
        try:
            while True:
                to_load = pickle.load(f)
                allready_done.add(to_load[0])
        except EOFError:
            pass
    dict_to_export = {
        "id" : f"{game}",
        "name" : f"{to_load[1]}",
        "startdate" : None,
        "enddate" : None,
        "pandas" : None,
        "wordclouds": {0:[]}
    }
    
    intermediate2 = copy.deepcopy(to_load[4])
    intermediate3 = copy.deepcopy(to_load[3])

    
    complete = intermediate2.merge(intermediate3, how='outer', on='created')
    complete["review_id_x"] = complete["review_id_x"].apply(lambda x: x if isinstance(x, list) else [])
    complete["review_id_y"] = complete["review_id_y"].apply(lambda x: x if isinstance(x, list) else [])
    nummerische_cols = ["positive_x", "negative_x", "written_during_early_access_x", "positive_y", "negative_y", "written_during_early_access_y"]
    complete[nummerische_cols] = complete[nummerische_cols].fillna(0).astype(int)
    complete["early_acces"] = complete["written_during_early_access_x"].abs() + complete["written_during_early_access_y"].abs()
    complete["early_acces"] = (complete["early_acces"] != 0).astype(int)
    complete["missing"] = complete["review_id_x"].map(set) - complete["review_id_y"].map(set)
    complete["positiv"] = complete[["positive_x", "positive_y"]].max(axis=1)
    complete["negativ"] = complete[["negative_x", "negative_y"]].min(axis=1)
    complete["missing"] = complete["missing"].apply(list)
    
    
    complete = complete.drop(columns = nummerische_cols)
    complete = complete.drop(columns = ["review_id_x", "review_id_y"])
    
    complete['created'] = pd.to_datetime(complete['created'])
    complete = complete.sort_values(by='created').reset_index(drop=True)
    
    earliest_date = complete["created"][0]
    last_date = complete["created"][len(complete["created"])-1]
    idx = pd.date_range(earliest_date, last_date)

    
    complete.index = pd.DatetimeIndex(complete["created"])
    complete = complete.reindex(idx)
    complete["created"] = complete.index
    complete = complete.reset_index(drop=True)
    
    complete[["positiv", "negativ", "early_acces"]] = complete[["positiv", "negativ", "early_acces"]].fillna(0).astype(int)
    complete["missing"] = complete["missing"].apply(
        lambda x: x if isinstance(x, list) else list()
    )
    complete["early_acces"] = (
        complete["early_acces"][::-1].cummax()[::-1]
    )
    
    complete["created"] = complete["created"].map(lambda dt: dt.strftime('%Y-%m'))    
    grouped = complete.groupby("created").agg({
        "positiv": "sum",
        "negativ": "sum",
        "early_acces": "min",
        "missing": lambda x: set().union(*x)
    })
    grouped = grouped.sort_values(by='created')
    grouped["created"] = grouped.index
    grouped = grouped.reset_index(drop=True)
    #print("10")
    #print(grouped)

    grouped["missing_fs"] = grouped["missing"].apply(
        lambda x: frozenset(x) if isinstance(x, (list, set)) else frozenset()
    )
    
    grouped["wordcloud_key"] = 0
    
    wordcloud_dict = {0: set()}
    
    current_key = 0
    current_missing = set()
    
    prev_date = None
    
    for idx, row in grouped.iterrows():
    
        current_date = pd.to_datetime(row["created"])
    
        # leerer Monat -> kein Wordcloud Block
        if len(row["missing_fs"]) == 0:
            prev_date = None
            current_missing = set()
            continue
    
        # neuer Block starten
        if (
            prev_date is None or
            (
                (current_date.year - prev_date.year) * 12
                + (current_date.month - prev_date.month)
            ) > 1
        ):
    
            current_key += 1
            current_missing = set(row["missing_fs"])
    
        # gleicher zusammenhängender Block
        else:
            current_missing.update(row["missing_fs"])
    
        grouped.at[idx, "wordcloud_key"] = current_key
    
        wordcloud_dict[current_key] = set(current_missing)
    
        prev_date = current_date
    #print("40")
    #print(grouped)
    grouped = grouped.drop(columns = ["missing", "missing_fs"])
    #print("50")
    #print(grouped)

    # grouped.to_json(f'./newjsons/game_{game}_combined.json', index=True)

    newdict = {0:[]}
    
    if f"text_{game}.pk" in files:
        testdicts = list()
        with open(f"./outputs/text_{game}.pk", "rb") as f:
            try:
                while True:
                    to_load = pickle.load(f)
                    #textdict = {k: to_load[k] for k in wordcloud_dict[1] if k in to_load}
                    testdicts.append(to_load)
            except EOFError:
                pass


        #print(testdicts[0])      
            
        #for textdict in testdicts:
        #    selected_keys = textdict.keys()
        #    # Texte zusammenführen
        #    selected_texts = " ".join(
        #    textdict[key] for key in selected_keys if key in textdict and textdict[key].strip()
        #    )
        
        for cloud_key in wordcloud_dict.keys():
            ngram_min = 2
            ngram_max = 2
            #print(wordcloud_dict[cloud_key])
            #selected_keys = textdict.keys()
            selected_keys = {k: testdicts[0][k] for k in wordcloud_dict[cloud_key] if k in testdicts[0]}.keys()
            selected_texts = " ".join(
            testdicts[0][key] for key in selected_keys if key in testdicts[0] and testdicts[0][key].strip()
            )
            #print(selected_keys)
            #print(selected_texts)
            #selected_texts = re.sub(r'<[^>]+>', ' ', selected_texts)
            #selected_texts = re.sub(r'\[/?[a-zA-Z0-9\\*\']+\]', ' ', selected_texts)
            #selected_texts = re.sub(r'[^A-Za-zÄÖÜäöüß❤\\*\' ]+', ' ', selected_texts)
            #selected_texts = re.sub(r"[^A-Za-zÄÖÜäöüß❤0-9 \*']+", ' ', selected_texts)
            selected_texts = re.sub(r'<[^>]+>', ' ', selected_texts)
            selected_texts = re.sub(
                r"\[/?[a-zA-Z0-9*']+\]",
                ' ',
                selected_texts
            )
            selected_texts = re.sub(
                r"[^A-Za-zÄÖÜäöüß❤ *']+",
                ' ',
                selected_texts
            )
            selected_texts = re.sub(
                re.escape(dict_to_export["name"]),
                " ",
                selected_texts,
                flags=re.IGNORECASE
            )
            vectorizer = CountVectorizer(
                stop_words=sorted(all_stopwords),
                ngram_range=(ngram_min, ngram_max)
            )
            tokens = [
                w for w in selected_texts.split()
                if w.lower() not in all_stopwords
            ]
            
            if len(tokens) < ngram_min:
                dict_to_export["wordclouds"][cloud_key] = {"Keine relevanten Reviews":1}
                #pass
                #print("Keine relevanten Wörter vorhanden")
            else:
                #print(selected_texts)
                #print(tokens)
                X = vectorizer.fit_transform([selected_texts])
                
                words = vectorizer.get_feature_names_out()
                counts = X.toarray().sum(axis=0)
                
                freq_dict = dict(zip(words, counts))
                
                wordcloud = WordCloud(
                    width=1200,
                    height=600,
                    background_color='black',
                    colormap='viridis'
                ).generate_from_frequencies(freq_dict)
                # Anzeigen
                plt.figure(figsize=(15, 7))
                plt.imshow(wordcloud, interpolation='bilinear')
                plt.axis('off')
                plt.title(f"{game} - {cloud_key}")
                plt.show()
                clean_words = {
                    k: float(v)
                    for k, v in wordcloud.words_.items()
                }
                dict_to_export["wordclouds"][cloud_key] = clean_words
                #print(clean_words)
        #print(dict_to_export)
    
    #dict_to_export["startdate"] = earliest_date
    #dict_to_export["enddate"] = last_date
    #dict_to_export["pandas"] = grouped.to_json(index=True)
    dict_to_export["startdate"] = str(earliest_date)
    dict_to_export["enddate"] = str(last_date)
    dict_to_export["pandas"] = grouped.to_dict(orient="index")

    with open(f"./json/{game}.json", "w", encoding="utf-8") as f:
        json.dump(
            dict_to_export,
            f,
            ensure_ascii=False,
            indent=2
        )